In [0]:
import mlflow, mlflow.sklearn, numpy as np
from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

mlflow.set_registry_uri("databricks-uc")

gt = spark.table("workspace.equipment.gold_train_features")
base = [c for c in gt.columns if c.endswith("_avg5")]
raw = [c[:-5] for c in base]
FEATS = ["cycle"] + raw + base

train_pd = gt.select(*FEATS, "rul_capped").toPandas()
test_pd = (spark.table("workspace.equipment.gold_test_features")
           .filter("cycle = max_cycle").orderBy("unit")
           .select("unit", *FEATS, "rul").toPandas())

X, y = train_pd[FEATS], train_pd["rul_capped"]
Xt, yt = test_pd[FEATS], np.minimum(test_pd["rul"], 125)

with mlflow.start_run(run_name="rf_rul_fd001"):
    model = RandomForestRegressor(n_estimators=200, max_depth=14, min_samples_leaf=3,
                                  n_jobs=-1, random_state=42)
    model.fit(X, y)
    pred = model.predict(Xt)
    rmse = float(np.sqrt(mean_squared_error(yt, pred)))
    mae = float(mean_absolute_error(yt, pred))
    mlflow.log_params({"n_estimators": 200, "max_depth": 14, "rul_cap": 125, "features": len(FEATS)})
    mlflow.log_metrics({"rmse": rmse, "mae": mae})
    mlflow.sklearn.log_model(model, "model",
                             signature=infer_signature(X, model.predict(X)),
                             input_example=X.head(3),
                             registered_model_name="workspace.equipment.rul_model")
    print(f"RMSE: {rmse:.2f} | MAE: {mae:.2f}")

health = test_pd[["unit", "cycle"]].copy()
health["predicted_rul"] = np.round(pred, 1)
health["true_rul"] = test_pd["rul"].values
health["abs_error"] = (health["predicted_rul"] - np.minimum(health["true_rul"], 125)).abs()
health["status"] = np.where(health["predicted_rul"] <= 30, "Critical",
                    np.where(health["predicted_rul"] <= 60, "Warning", "Healthy"))
spark.createDataFrame(health).write.mode("overwrite").option("overwriteSchema", "true") \
     .saveAsTable("workspace.equipment.gold_engine_health")

spark.sql("SELECT status, COUNT(*) AS engines, ROUND(AVG(abs_error),1) AS avg_error "
          "FROM workspace.equipment.gold_engine_health GROUP BY status").show()